# Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

# Training the Image Captioning Model

This notebook trains a **CNN-LSTM image captioning model** using the MS COCO dataset.

The model follows an **Encoder-Decoder architecture**:

- A **CNN Encoder** extracts high-level visual features from an input image.
- An **LSTM Decoder** uses the encoded image features and caption embeddings to generate a caption one word at a time.

During training, each image is paired with a ground-truth caption. The image is first passed through the CNN encoder to obtain a visual representation. The caption tokens are converted into word embeddings and processed by the LSTM decoder.

At each time step, the decoder produces logits over the entire vocabulary. The model is trained by minimizing the **cross-entropy loss** between the predicted vocabulary logits and the ground-truth caption tokens.

---

## 1. Training Configuration

The following hyperparameters are used for the initial training experiment:

| Hyperparameter | Value | Description |
|---|---:|---|
| `batch_size` | 32 | Number of image-caption samples processed in each batch |
| `vocab_threshold` | 5 | Minimum word frequency required for inclusion in the vocabulary |
| `vocab_from_file` | True | Load the previously generated vocabulary from file |
| `embed_size` | 256 | Dimensionality of the image and word embeddings |
| `hidden_size` | 256 | Number of features in the LSTM hidden state |
| `num_epochs` | 3 | Number of complete passes through the training dataset |
| `save_every` | 1 | Save model checkpoints after every epoch |
| `print_every` | 100 | Report training statistics every 100 iterations |
| `learning_rate` | 0.001 | Learning rate used by the Adam optimizer |

For this initial version, the training pipeline uses a subset of **2,000 caption samples**. This reduces training time while allowing the complete data loading, forward propagation, loss computation, backpropagation, and checkpointing pipeline to be verified.

---

## 2. Image Preprocessing

Before being passed to the CNN encoder, each training image is processed using the following transformation pipeline:

1. **Resize** the image so that its smaller edge is 256 pixels.
2. **Randomly crop** a `224 × 224` region.
3. Convert the image into a **PyTorch tensor**.
4. **Normalize** the RGB channels using the mean and standard deviation expected by the pretrained CNN.

The normalization values are:

```python
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)
```

The random crop introduces variation into the training images and provides a simple form of data augmentation.

Normalization ensures that the image distribution is compatible with the pretrained CNN used by the encoder.

---

## 3. Model Architecture

The model contains two main components:

### CNN Encoder

The encoder extracts high-level visual features from each image and projects them into a **256-dimensional embedding space**.

```python
encoder = EncoderCNN(embed_size=256)
```

### LSTM Decoder

The decoder receives the encoded image representation and caption embeddings.

```python
decoder = DecoderRNN(
    embed_size=256,
    hidden_size=256,
    vocab_size=vocab_size
)
```

The LSTM models the sequential relationship between words in the caption.

At each time step, the decoder produces a vector of size `vocab_size`, representing the logits for all possible words in the vocabulary.

---

## 4. Trainable Parameters

All parameters in the decoder are trainable.

For the encoder, only parameters with `requires_grad=True` are included in the optimization process.

```python
params = (
    list(decoder.parameters()) +
    list(filter(lambda p: p.requires_grad, encoder.parameters()))
)
```

This approach allows the model to take advantage of **transfer learning**.

The pretrained CNN can retain previously learned visual representations while the trainable encoder components and the decoder adapt to the image captioning task.

This reduces computational cost compared with training the entire CNN from scratch.

---

## 5. Loss Function

The model uses **Cross-Entropy Loss** as the training objective.

```python
criterion = nn.CrossEntropyLoss()
```

At each caption time step, the decoder predicts logits over the entire vocabulary.

Cross-entropy loss compares these predictions with the corresponding ground-truth caption tokens.

The training objective is therefore to increase the probability assigned to the correct next word in the caption sequence.

---

## 6. Optimizer

The model is trained using the **Adam optimizer** with a learning rate of `0.001`.

```python
optimizer = torch.optim.Adam(
    params=params,
    lr=0.001
)
```

Adam adaptively updates the learning rate for each trainable parameter based on gradient statistics.

This makes it a practical optimizer for training neural networks containing embedding and recurrent layers.

---

## 7. Device Configuration

The training pipeline automatically uses a GPU when CUDA is available.

```python
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
```

Both the encoder and decoder are moved to the selected device before training.

This allows the same training code to run in both GPU-enabled and CPU-only environments.

---

## 8. Training Pipeline

The overall training pipeline can be summarized as:

**Image → Image Preprocessing → CNN Encoder → Image Embedding → LSTM Decoder → Vocabulary Logits → Cross-Entropy Loss → Backpropagation → Adam Update**

For each training batch:

1. Load a batch of images and their corresponding captions.
2. Apply the image preprocessing pipeline.
3. Pass the images through the CNN encoder.
4. Convert caption tokens into word embeddings.
5. Pass the image features and caption embeddings through the LSTM decoder.
6. Generate vocabulary logits at each sequence step.
7. Compute the cross-entropy loss.
8. Perform backpropagation.
9. Update the trainable model parameters using Adam.
10. Periodically report training loss and perplexity.
11. Save model checkpoints after each epoch.

The resulting checkpoints can later be loaded by the inference pipeline to generate captions for unseen images.

In [9]:
%load_ext autoreload
%autoreload 2
import math

import sys
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms

# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
COCO_ROOT = (Path.cwd() / "../../COCOapi").resolve()

# Allow imports from src/
sys.path.insert(0, str(PROJECT_ROOT))

# --------------------------------------------------
# Project modules
# --------------------------------------------------

from src.data_loader import get_loader
from src.model import EncoderCNN, DecoderRNN

# --------------------------------------------------
# Training configuration
# --------------------------------------------------

batch_size = 32
vocab_threshold = 5
vocab_from_file = True

embed_size = 256
hidden_size = 256

num_epochs = 3
save_every = 1
print_every = 100

learning_rate = 0.001
log_file = "training_log.txt"


# --------------------------------------------------
# Image preprocessing
# --------------------------------------------------

transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])


# -------------------------------------------
# Data loader
# -------------------------------------------

data_loader = get_loader(
    transform=transform_train,
    mode="train",
    batch_size=batch_size,
    vocab_threshold=vocab_threshold,
    vocab_from_file=vocab_from_file,
    cocoapi_loc=str(COCO_ROOT),
)

vocab_size = len(data_loader.dataset.vocab)

print("Vocabulary size:", vocab_size)


# --------------------------------------------------
# Model
# --------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderRNN(
    embed_size,
    hidden_size,
    vocab_size,
).to(device)


# --------------------------------------------------
# Loss and optimizer
# --------------------------------------------------

criterion = nn.CrossEntropyLoss()

params = (
    list(decoder.parameters())
    + [p for p in encoder.parameters() if p.requires_grad]
)

optimizer = torch.optim.Adam(
    params=params,
    lr=learning_rate,
)


# --------------------------------------------------
# Training steps
# --------------------------------------------------

total_step = math.ceil(
    len(data_loader.dataset.caption_lengths)
    / data_loader.batch_sampler.batch_size
)

print(f"Device: {device}")
print(f"Vocabulary size: {vocab_size}")
print(f"Training steps per epoch: {total_step}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Vocabulary successfully loaded from ./vocab.pkl.
loading annotations into memory...
Done (t=0.26s)
creating index...
index created!
Obtaining caption lengths...


100%|██████████| 414113/414113 [00:07<00:00, 52000.21it/s]


Vocabulary size: 8852
Device: mps
Device: mps
Vocabulary size: 8852
Training steps per epoch: 12942


## Step 2: Train the Image Captioning Model

In this step, we train the CNN-RNN image captioning model using the COCO training dataset.

The training pipeline combines two components:

- **CNN Encoder** — extracts visual features from each input image using a pretrained ResNet backbone.
- **RNN Decoder** — takes the encoded image representation and predicts the corresponding caption token by token.

During training, the model performs the following pipeline:

**Image → CNN Encoder → Image Embedding → RNN Decoder → Vocabulary Scores**

For each training batch:

1. Images and their corresponding captions are loaded from the training dataset.
2. Images are passed through the CNN encoder to obtain compact visual feature representations.
3. Caption tokens are converted into word embeddings.
4. The image features and caption embeddings are processed by the RNN decoder.
5. The decoder produces a vocabulary score for each position in the caption sequence.
6. Cross-entropy loss is computed between the predicted vocabulary scores and the ground-truth caption tokens.
7. Gradients are propagated backward through the trainable parameters.
8. The optimizer updates the model parameters.

Training loss and perplexity are recorded throughout training to monitor convergence.

### Training Objective

Given an image and its ground-truth caption, the model learns to maximize the probability of the correct token at each position in the caption sequence.

The decoder output has the shape:

`[batch_size, sequence_length, vocab_size]`

where each vector along the final dimension contains the predicted scores for all words in the vocabulary.

The training objective is optimized using cross-entropy loss.

### Checkpointing

Model checkpoints are saved periodically during training so that trained weights can be reused for inference or additional training.

Each checkpoint contains the learned parameters of the encoder and decoder.

### Training Configuration

The main training hyperparameters include:

- Batch size
- Image/word embedding dimension
- RNN hidden-state dimension
- Learning rate
- Number of epochs
- Vocabulary frequency threshold

These parameters can be adjusted to study their effect on training convergence and caption quality.

In [ ]:
import os
import sys

import numpy as np
import torch
import torch.utils.data as data


# Create checkpoint directory.
checkpoint_dir = "./models"
os.makedirs(checkpoint_dir, exist_ok=True)

# Open training log.
with open(log_file, "w") as log_f:

    accumulation_steps = 4

    optimizer.zero_grad()

    #max_steps = 50

    for epoch in range(1, num_epochs + 1):

        encoder.train()
        decoder.train()

        for i_step in range(1, total_step + 1):

            #if i_step >= max_steps:
                #break

            # Sample captions with the same sequence length.
            indices = data_loader.dataset.get_train_indices()

            sampler = data.sampler.SubsetRandomSampler(
                indices=indices
            )

            data_loader.batch_sampler.sampler = sampler

            # Load one training batch.
            images, captions = next(iter(data_loader))

            # Move tensors to the selected device.
            images = images.to(device)
            captions = captions.to(device)

            # Forward pass.
            features = encoder(images)
            outputs = decoder(features, captions)

            # Compute token-level cross-entropy loss.
            loss = criterion(
                outputs.reshape(-1, vocab_size),
                captions.reshape(-1),
            )

            # Scale loss for gradient accumulation.
            scaled_loss = loss / accumulation_steps

            # Backward pass.
            scaled_loss.backward()

            # Update model parameters after accumulating gradients.
            if (
                i_step % accumulation_steps == 0
                or i_step == total_step
            ):
                optimizer.step()
                optimizer.zero_grad()

            # Training statistics.
            loss_value = loss.item()
            perplexity = np.exp(loss_value)

            stats = (
                f"Epoch [{epoch}/{num_epochs}], "
                f"Step [{i_step}/{total_step}], "
                f"Loss: {loss_value:.4f}, "
                f"Perplexity: {perplexity:.4f}"
            )

            print("\r" + stats, end="")
            sys.stdout.flush()

            log_f.write(stats + "\n")
            log_f.flush()

            if i_step % print_every == 0:
                print()

        # Save checkpoints.
        if epoch % save_every == 0:
            torch.save(
                decoder.state_dict(),
                os.path.join(
                    checkpoint_dir,
                    f"decoder-{epoch}.pt"
                ),
            )

            torch.save(
                encoder.state_dict(),
                os.path.join(
                    checkpoint_dir,
                    f"encoder-{epoch}.pt"
                ),
            )

Epoch [1/3], Step [2/12942], Loss: 3.9667, Perplexity: 52.8126

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here. 

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.

In [ ]:
# (Optional) TODO: Validate your model.